# ClinVar: collapse long-tail CLNSIG strings

ClinVar `CLNSIG` values are notoriously long-tail — the same biological
interpretation gets encoded in dozens of slightly different
combinations. `biodb.clinvar.simplify_annotations` collapses that
vocabulary into 6 (and then 4) buckets:

- **6-class** (`CLNSIG_simple`): benign / likely_benign / path /
  likely_path / conflicting / other
- **4-class** (`CLNSIG_super_simple`): benign / path / conflicting /
  other

This notebook uses a synthetic mini-table — no NCBI download needed.

In [1]:
import polars as pl

from biodb import clinvar

# A handful of real CLNSIG strings observed in ClinVar.
df = pl.DataFrame(
    {
        "CLNSIG": [
            "Benign",
            "Likely_benign",
            "Pathogenic",
            "Likely_pathogenic",
            "Pathogenic/Likely_pathogenic",
            "Conflicting_classifications_of_pathogenicity",
            "Uncertain_significance",
            "Pathogenic|drug_response",
            "Benign|risk_factor",
        ],
        "GENEINFO": [
            "BRCA1:672",
            "TP53:7157",
            "EGFR:1956",
            "MYC:4609",
            "PTEN:5728",
            "KRAS:3845",
            "APOE:348",
            "CYP2C19:1557",
            "MTHFR:4524",
        ],
    }
)
df

CLNSIG,GENEINFO
str,str
"""Benign""","""BRCA1:672"""
"""Likely_benign""","""TP53:7157"""
"""Pathogenic""","""EGFR:1956"""
"""Likely_pathogenic""","""MYC:4609"""
"""Pathogenic/Likely_pathogenic""","""PTEN:5728"""
"""Conflicting_classifications_of…","""KRAS:3845"""
"""Uncertain_significance""","""APOE:348"""
"""Pathogenic|drug_response""","""CYP2C19:1557"""
"""Benign|risk_factor""","""MTHFR:4524"""


In [2]:
out = clinvar.simplify_annotations(df, verbose=True)
out.select(["CLNSIG", "CLNSIG_simple", "CLNSIG_super_simple", "GENE"])

Using default maps.
Simplifying annotations.


CLNSIG,CLNSIG_simple,CLNSIG_super_simple,GENE
str,str,str,str
"""Benign""","""benign""","""benign""","""BRCA1"""
"""Likely_benign""","""likely_benign""","""benign""","""TP53"""
"""Pathogenic""","""path""","""path""","""EGFR"""
"""Likely_pathogenic""","""likely_path""","""path""","""MYC"""
"""Pathogenic/Likely_pathogenic""","""likely_path""","""path""","""PTEN"""
"""Conflicting_classifications_of…","""conflicting""","""conflicting""","""KRAS"""
"""Uncertain_significance""","""other""","""other""","""APOE"""
"""Pathogenic|drug_response""","""path""","""path""","""CYP2C19"""
"""Benign|risk_factor""","""benign""","""benign""","""MTHFR"""


Unknown strings fall through to ``other`` rather than raising — useful
for ad-hoc filtering and group-by operations on raw ClinVar exports.

In [3]:
unknown = pl.DataFrame({"CLNSIG": ["mystery_string_not_in_map"]})
clinvar.simplify_annotations(unknown, verbose=False)

CLNSIG,CLNSIG_simple,CLNSIG_super_simple
str,str,str
"""mystery_string_not_in_map""","""other""","""other"""


## Filtering a parsed ClinVar table

`filter_df` accepts column → value pairs. List values become
OR-substring matches; the special `CLNREVSTAT_score` key is compared
as `>=`; everything else is equality.

In [4]:
sample = pl.DataFrame(
    {
        "CHROM": ["1", "1", "2", "3"],
        "POS": [10, 20, 30, 40],
        "CLNDN": ["a", "b", "c", "d"],
        "CLNREVSTAT_score": [0, 1, 2, 4],
        "CLNVC": [
            "single_nucleotide_variant",
            "single_nucleotide_variant",
            "Deletion",
            "single_nucleotide_variant",
        ],
    }
)
sample

CHROM,POS,CLNDN,CLNREVSTAT_score,CLNVC
str,i64,str,i64,str
"""1""",10,"""a""",0,"""single_nucleotide_variant"""
"""1""",20,"""b""",1,"""single_nucleotide_variant"""
"""2""",30,"""c""",2,"""Deletion"""
"""3""",40,"""d""",4,"""single_nucleotide_variant"""


In [5]:
clinvar.filter_df(
    sample,
    filters={
        "CLNREVSTAT_score": 2,  # >= 2 (multi-submitter consensus or stronger)
        "CLNVC": "single_nucleotide_variant",
    },
)

Filtered DataFrame shape: (1, 5)
Variant count: 1


CHROM,POS,CLNDN,CLNREVSTAT_score,CLNVC
str,i64,str,i64,str
"""3""",40,"""d""",4,"""single_nucleotide_variant"""
